In [22]:
import torch
from torch import nn
from d2l import torch as d2l

In [ ]:
# Batch Normalization 계산

# Batch Normalization은 intermediate variable을 지속적으로 
# center하고 rescaling하기 때문에 training 중 magnitude가 끝없이 발산하는 것을 방지
def batch_norm(
    # Input batch
    X: torch.Tensor,
    
    # 분포 복원용 learnable params
    gamma: torch.Tensor,
    beta: torch.Tensor,
    
    # 누적한 평균/분산
    moving_mean: torch.Tensor,
    moving_var: torch.Tensor,
    
    # Division by zero를 방지
    eps: float,
    
    # 현재 minibatch statistics를 
    # moving statistics에 반영하는 비율
    momentum: float,
) -> tuple[
    torch.Tensor,
    torch.Tensor,
    torch.Tensor,
]:
    
    # [Prediction mode]:
    # moving value를 update 없이 그대로 사용
    if not torch.is_grad_enabled():
        X_hat = (
            X - moving_mean
        ) / torch.sqrt(
            moving_var + eps
        )
        
        
        
    # [Training mode]:
    # 현재 minibatch의 mean과 variance를 직접 계산하여 X를 Normalization
    else:
        
        if X.ndim not in (2, 4):
            raise ValueError(
                "X must be a 2D or 4D tensor."
            )
            
        # 1) Fully Connected Layer: 같은 feature를 batch 전체에서 모음
        # input shape : [B, F]
        # mean shape  : [F]
        # variance    : [F]
        if X.ndim == 2:
            
            mean = X.mean(
                dim=0,
            )
            
            var = (
                (X - mean) ** 2
            ).mean(
                dim=0,
            )
            
            
        # 2) Convolution Layer: 같은 channel을 batch와 모든 spatial location에서 모음
        # 각 Channel을 독립적으로 Normalization.
        # input shape    : [B, C, H, W]
        # mean shape     : [1, C, 1, 1]
        # variance shape : [1, C, 1, 1]
        else: 
            
            mean = X.mean(
                dim=(0, 2, 3),
                keepdim=True,
            )
            
            var = (
                (X - mean) ** 2
            ).mean(
                dim=(0, 2, 3),
                keepdim=True,
            )
            
            
        # Standardization: (mean=0, var=1)
        X_hat = (
            X - mean
        ) / torch.sqrt(
            var + eps
        )
        
        # 새로 계산된 값을 momentum 만큼 반영
        moving_mean = (
            (1.0 - momentum) * moving_mean
            + momentum * mean
        )
        moving_var = (
            (1.0 - momentum) * moving_var
            + momentum * var
        )
        
    # Network가 필요한 scale과 shift를 학습하도록
    # learnable parameter gamma와 beta를 적용.
    Y = gamma * X_hat + beta
        
    return (
        Y,
        moving_mean.detach(),
        moving_var.detach(),
    )

In [ ]:
# Custom BatchNorm Layer

class BatchNorm(nn.Module):
    def __init__(
        self,
        
        # Fully Connected Layer -> feature 수
        # Convolution Layer -> output channel 수
        num_features: int,
        
        # X의 dimension 수: Fully Connected는 2, Convolution은 4
        num_dims:int,       
    ) -> None:
        super().__init__()
        
        
        # [Fully Connected output]
        # X.shape     = [B, F]
        # gamma.shape = [1, F]
        # beta.shape  = [1, F]
        if num_dims == 2:
            shape = (
                1,
                num_features,
            )
            
        # [Convolution output]
        # X.shape     = [B, C, H, W]
        # gamma.shape = [1, C, 1, 1]
        # beta.shape  = [1, C, 1, 1]
        elif num_dims == 4:
            shape = (
                1,
                num_features,
                1,
                1,
            )
            
        else:
            raise ValueError(
                "num_dims must be 2 or 4."
            )
            
            
        # [Learnable Parameter Initialization]:
        # - Beta : offset param
        # - Gamma: scale param
        #  nn.Parameter 이므로 model.parameters()에 포함되며 optimizer가 update함.
        self.beta = nn.Parameter(
            torch.zeros(
                shape,
            )
        )
        self.gamma = nn.Parameter(
            torch.ones(
                shape,
            )
        )

        
        # [Moving Statistics]
        #  moving mean & moving variance는 batch_norm()에서 momentum으로 update.
        #  Prediction에서 누적된 값이 사용된다.
        self.moving_mean: torch.Tensor = torch.zeros(
            shape,
        )
        self.moving_var: torch.Tensor = torch.ones(
            shape,
        )
        
        
    def forward(
        self,
        X: torch.Tensor,
    ) -> torch.Tensor:
        if self.moving_mean.device != X.device:
            self.moving_mean = self.moving_mean.to(
                X.device
            )

            self.moving_var = self.moving_var.to(
                X.device
            )

        Y, self.moving_mean, self.moving_var = batch_norm(
            X,
            self.gamma,
            self.beta,
            self.moving_mean,
            self.moving_var,
            eps=1e-5,
            momentum=0.1,
        )

        return Y

In [25]:
# Fully Connected Batch Normalization

# Input shape: [B, F] = [8, 3]
fully_connected_X = (
    torch.randn(
        8,
        3,
    )
    * 5
    + 10
)

fully_connected_bn = BatchNorm(
    num_features=3,
    num_dims=2,
)

# Gradient가 활성화되어 있으므로 Training 분기로 진입
fully_connected_Y = fully_connected_bn(
    fully_connected_X
)

print(
    "Input shape:",
    tuple(fully_connected_X.shape),
)

print(
    "Output shape:",
    tuple(fully_connected_Y.shape),
)

# 각 feature를 batch dimension인 dim=0을 따라 확인
print(
    "\nFeature mean before BN:",
    fully_connected_X.mean(
        dim=0,
    ),
)

print(
    "Feature mean after BN:",
    fully_connected_Y.mean(
        dim=0,
    ),
)

print(
    "\nFeature variance before BN:",
    fully_connected_X.var(
        dim=0,
        unbiased=False,
    ),
)

print(
    "Feature variance after BN:",
    fully_connected_Y.var(
        dim=0,
        unbiased=False,
    ),
)

print(
    "\nGamma:",
    fully_connected_bn.gamma,
)

print(
    "Beta:",
    fully_connected_bn.beta,
)

print(
    "Moving mean:",
    fully_connected_bn.moving_mean,
)

print(
    "Moving variance:",
    fully_connected_bn.moving_var,
)

Input shape: (8, 3)
Output shape: (8, 3)

Feature mean before BN: tensor([ 8.2096, 11.1213, 10.2038])
Feature mean after BN: tensor([ 8.9407e-08,  1.3411e-07, -5.5879e-08], grad_fn=<MeanBackward1>)

Feature variance before BN: tensor([28.4193, 34.8218, 35.6026])
Feature variance after BN: tensor([1.0000, 1.0000, 1.0000], grad_fn=<VarBackward0>)

Gamma: Parameter containing:
tensor([[1., 1., 1.]], requires_grad=True)
Beta: Parameter containing:
tensor([[0., 0., 0.]], requires_grad=True)
Moving mean: tensor([[0.8210, 1.1121, 1.0204]])
Moving variance: tensor([[3.7419, 4.3822, 4.4603]])


In [ ]:
# Convolution Batch Normalization

# Input shape: [B, C, H, W] = [2, 3, 4, 4]
convolution_X = torch.randn(
    2,
    3,
    4,
    4,
)

convolution_bn = BatchNorm(
    num_features=3,
    num_dims=4,
)

# Gradient가 활성화되어 있으므로 Training 분기로 진입
convolution_Y = convolution_bn(
    convolution_X
)

print(
    "Input shape:",
    tuple(convolution_X.shape),
)

print(
    "Output shape:",
    tuple(convolution_Y.shape),
)

# 각 channel에 대해 B, H, W dimension을 줄인다.
# dim=1인 channel dimension만 남으므로
# channel별 mean과 variance를 확인 가능
print(
    "\nChannel mean after BN:",
    convolution_Y.mean(
        dim=(0, 2, 3),
    ),
)

print(
    "Channel variance after BN:",
    convolution_Y.var(
        dim=(0, 2, 3),
        unbiased=False,
    ),
)

print(
    "\nGamma shape:",
    tuple(convolution_bn.gamma.shape),
)

print(
    "Beta shape:",
    tuple(convolution_bn.beta.shape),
)

print(
    "Moving mean shape:",
    tuple(convolution_bn.moving_mean.shape),
)

print(
    "Moving variance shape:",
    tuple(convolution_bn.moving_var.shape),
)

Input shape: (2, 3, 4, 4)
Output shape: (2, 3, 4, 4)

Channel mean after BN: tensor([1.8626e-08, 0.0000e+00, 0.0000e+00], grad_fn=<MeanBackward1>)
Channel variance after BN: tensor([1.0000, 1.0000, 1.0000], grad_fn=<VarBackward0>)

Gamma shape: (1, 3, 1, 1)
Beta shape: (1, 3, 1, 1)
Moving mean shape: (1, 3, 1, 1)
Moving variance shape: (1, 3, 1, 1)


In [ ]:
# Prediction에서 Moving Statistics 사용 확인

# Prediction 전에 현재 moving statistics를 복사한다.
moving_mean_before = (
    convolution_bn.moving_mean.clone()
)

moving_var_before = (
    convolution_bn.moving_var.clone()
)


# torch.no_grad()에서는 Gradient가 비활성화되므로
# batch_norm function의 Prediction 분기로 진입.
# 현재 input의 mean과 variance를 계산하지 않고
# 저장된 moving statistics를 사용한다.
with torch.no_grad():
    prediction_Y = convolution_bn(
        convolution_X
    )

print(
    "Prediction output shape:",
    tuple(prediction_Y.shape),
)

# Prediction에서는 moving statistics를 update X
print(
    "Moving mean unchanged:",
    torch.equal(
        moving_mean_before,
        convolution_bn.moving_mean,
    ),
)

print(
    "Moving variance unchanged:",
    torch.equal(
        moving_var_before,
        convolution_bn.moving_var,
    ),
)

Prediction output shape: (2, 3, 4, 4)
Moving mean unchanged: True
Moving variance unchanged: True
